# 강의 04 · 실습 8 — 이미지 생성 파이프라인 · (5) 고난도 2 — 실패 유형 구분과 안내


## 1. 문제상황

- 이미지 모델 호출은 가끔 실패합니다.
- 서버가 잠시 응답하지 않는 실패는 조금 뒤 다시 부르면 대개 성공합니다.
- 안전 정책으로 차단된 실패는 몇 번을 다시 불러도 같은 결과가 나오므로, 다시 부르지 말고 사용자에게 안내해야 합니다.
- 지금은 두 실패가 같은 오류 화면으로 보여서, 디자이너가 오류 문구를 읽고 손으로 판단해 다시 부르거나 포기합니다.
- 다시 부를 실패를 포기하면 결과를 잃고, 포기할 실패를 다시 부르면 비용만 씁니다.

## 2. 문제와 목표

- **문제**: 실패의 종류를 사람이 읽어 판단합니다. 종류마다 해야 할 일이 정해져 있는데도 코드에 적혀 있지 않습니다.
- **목표**: 생성 노드가 호출 실패를 붙잡아 종류를 상태에 쓰고, 일시 오류이면 정해진 횟수 안에서 다시 부르고, 차단이면 다시 부르지 않고 사용자에게 안내한 뒤 끝내는 처리 흐름을 만듭니다. 정상이면 후보를 사람에게 보내 판정을 받습니다. 사람이 「재설계」라고 답하면 지시문 설계부터 다시 돕니다.
    - 실패 두 종류: 「일시」는 첫 호출에서만 연결 오류를 내고 그다음 호출부터는 정상으로 돌아오는 실패, 「차단」은 부를 때마다 권한 오류를 내는 실패입니다. 장애는 코드 안의 스위치 하나로 모의로 만들고, 실제 안전 필터는 건드리지 않습니다.
    - 정해진 횟수: 생성 재시도 상한 2회입니다.
    - 지시문 템플릿은 한 가지이며 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있고, 주제는 「비 내리는 밤 서울 골목의 LP 바 창가」입니다. 일시 오류와 차단은 각각 다른 `thread_id`로 실행하며, 사람의 답(「확정」 한 번)은 코드에 대본으로 미리 정해 넣습니다.
- **목표 달성 여부의 판정 기준**
    - 일시 오류를 모의한 실행에서는 이미지 생성이 두 번 시도되어 두 번째에 1장이 만들어진 뒤 사람의 판정을 기다리며 멈춥니다.
    - 차단을 모의한 실행에서는 생성이 한 번만 시도되고 다시 부르지 않은 채 사용자 안내를 거쳐 끝나는 것을 실행 결과에서 확인합니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex08_s5_diagram.svg)

## 4. 단계별 요구사항

(학생이 「3. 워크플로우 다이어그램」을 보고 번호 목록으로 씁니다.)

## 5. 코드 골격

이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다.

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 텍스트 모델과 이미지 모델을 준비합니다. 이미지 모델을 부르는 함수 `paint`도 여기서 정의합니다.

- API 키와 자격증명은 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다.
- `.env` 파일에는 다음 네 줄이 있어야 합니다. 값은 각자 발급받은 것을 넣습니다.

```
OPENAI_API_KEY=발급받은_키
GOOGLE_APPLICATION_CREDENTIALS=서비스_계정_키_파일의_경로
VERTEX_PROJECT=프로젝트_이름
VERTEX_LOCATION=리전_이름
```

- `paint` 호출 1회가 이미지 1장이고, 호출마다 비용이 듭니다. 생성한 이미지는 노트북 옆의 `out_images` 폴더에 저장됩니다.
- `show`는 후보 이미지 경로 목록을 화면에 표시하는 보조 함수입니다.
- 생성 호출 횟수는 전역 카운터 `paint_calls`로 셉니다.

In [ ]:
import os
import time
from operator import add
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from IPython.display import Image, display
from google import genai
from google.genai import types
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

load_dotenv(find_dotenv(usecwd=True))
for key in ("OPENAI_API_KEY", "GOOGLE_APPLICATION_CREDENTIALS", "VERTEX_PROJECT", "VERTEX_LOCATION"):
    if not os.environ.get(key):
        raise SystemExit(f"agentic-ai 폴더의 .env 파일에 {key} 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
client = genai.Client(vertexai=True, project=os.environ["VERTEX_PROJECT"], location=os.environ["VERTEX_LOCATION"])
IMG_MODEL = "gemini-3.1-flash-lite-image"
OUT_DIR = Path("out_images")
OUT_DIR.mkdir(exist_ok=True)
paint_calls = 0


def paint(prompt: str, tag: str) -> str:
    """지시문을 이미지 모델에 보내 이미지 파일을 저장하고 경로를 돌려준다. 호출 1회 = 이미지 1장 = 비용 발생."""
    global paint_calls
    for attempt in range(4):
        try:
            res = client.models.generate_content(
                model=IMG_MODEL, contents=prompt,
                config=types.GenerateContentConfig(response_modalities=["IMAGE", "TEXT"]))
            break
        except Exception as e:
            if "429" in str(e) and attempt < 3:
                print(f"    [paint] 분당 한도 초과 — {30 * (attempt + 1)}초 뒤 다시 부릅니다")
                time.sleep(30 * (attempt + 1))
                continue
            raise
    paint_calls += 1
    part = [p for p in res.candidates[0].content.parts if p.inline_data][0].inline_data
    ext = "png" if "png" in part.mime_type else "jpg"
    path = OUT_DIR / f"{tag}_{paint_calls:02d}.{ext}"
    path.write_bytes(part.data)
    print(f"    [paint] {path.as_posix()} ({len(part.data)} bytes)")
    return path.as_posix()


def show(paths: list) -> None:
    """후보 이미지 경로 목록을 화면에 차례로 표시한다."""
    for i, p in enumerate(paths, 1):
        print(f"    후보 {i}: {p}")
        display(Image(filename=p, width=320))


print("모델 준비를 마쳤습니다. 이미지 저장 폴더:", OUT_DIR)

# 주어진 자료 — 지시문 템플릿 (주제 문장을 뒤에 이어 붙여 언어 모델에 넣는다)
SPEC_V1 = ("다음 주제로 이미지 생성 지시문을 한 문단으로 쓴다. "
           "장면·조명·화각을 포함한다. 주제: ")


In [ ]:
# 여기에 코드를 작성합니다. 단계마다 셀을 나누어 작성합니다.


## 7. 실행 결과 확인

1. 일시 오류를 모의한 실행에서 생성이 두 번 시도되고, 두 번째에 이미지 1장이 만들어져 사람의 판정을 기다리며 멈춥니다. 「확정」을 답하면 끝납니다.
2. 차단을 모의한 실행에서 생성이 한 번만 시도되고, 다시 부르지 않은 채 사용자 안내를 거쳐 끝납니다. 이미지 파일은 만들어지지 않습니다.
3. 두 실행을 합쳐 이미지 생성 호출은 1회입니다. 실패의 종류를 코드가 구분했기 때문에 비용이 한 번만 들었습니다.
4. 실패의 종류와 시도 횟수는 상태에 남아 있어야 합니다. 다음 행선지를 고르는 것은 노드가 아니라 엣지입니다.